In [ ]:
!pip install ultralytics tensorflowjs

INFO: pip is looking at multiple versions of tf-keras to determine which version is compatible with other requirements. This could take a while.
INFO: pip is looking at multiple versions of wheel to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 26.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.1/89.1 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.0/53.0 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.1/16.1 MB 70.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 645.0/645.0 MB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 31.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 42.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 4.9 MB/s eta 0:00:00
  Attempting uninstall: wheel
    Found existing installation: wheel 0.47.0
    Uninstall

In [ ]:
# 1. Apaga a pasta dataset antiga inteira de forma forçada
!rm -rf /content/dataset

# 2. Descompacta o novo zip forçando a sobrescrita (-o) e em modo silencioso (-q)
!unzip -o -q /content/dataset.zip -d /content/dataset/

# 3. Lista os arquivos para confirmação
!ls /content/dataset/

data.yaml  README.dataset.txt  README.roboflow.txt  test  train  valid


In [ ]:
import os
from ultralytics import YOLO

# Configurações de Engenharia do TCC
DATA_YAML_PATH = '/content/dataset/data.yaml' # Caminho típico do Colab

# Métrica de Peso: Usaremos o modelo Nano real para rodar leve no mobile
MODEL_SIZE = 'yolov8s.pt' # Corrigido de 's' para 'n' (Nano)

EPOCHS = 100
IMG_SIZE = 640
BATCH_SIZE = 16

def treinar_modelo():
    print("🚀 Carregando arquitetura YOLOv8 Nano para o TCC...")
    model = YOLO(MODEL_SIZE)

    print(f"🔥 Iniciando treinamento pesado por {EPOCHS} épocas na GPU...")
    results = model.train(
        data=DATA_YAML_PATH,
        epochs=EPOCHS,
        imgsz=IMG_SIZE,
        batch=BATCH_SIZE,
        name='ALPR_Vistoria_YOLOv8n',
        device="cpu", # '0' força o uso da GPU no Google Colab (MUITO mais rápido)
    )

    print("\n📊 Analisando métricas de precisão...")
    map50 = results.results_dict['metrics/mAP50(B)']
    print(f"⭐ mAP@50: {map50:.4f}")

    # ==========================================
    # A CEREJA DO BOLO PARA O APLICATIVO REACT
    # ==========================================
    print("\n📦 Exportando modelo para a Web (TensorFlow.js)...")
    model.export(format='tfjs')
    print("✅ Sucesso! Pegue a pasta 'tfjs' gerada e coloque no seu React em public/model_yolo_placas/")

if __name__ == "__main__":
    treinar_modelo()

🚀 Carregando arquitetura YOLOv8 Nano para o TCC...
🔥 Iniciando treinamento pesado por 100 épocas na GPU...
Ultralytics 8.4.46 🚀 Python-3.12.13 torch-2.10.0+cpu CPU (Intel Xeon CPU @ 2.20GHz)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/dataset/data.yaml, degrees=0.0, deterministic=True, device=cpu, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8s.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, na

KeyboardInterrupt: 

In [ ]:
import os
from IPython.display import Image, display

# O caminho exato que configuramos no parâmetro 'name' do seu treinamento
pasta_resultados = '/content/runs/detect/ALPR_Vistoria_YOLOv8n-3'

print("📊 GERANDO RELATÓRIOS DE ARQUITETURA PARA O TCC...\n")

# 1. Gráfico Principal (Curvas de Aprendizado e Erro)
caminho_resultados = os.path.join(pasta_resultados, 'results.png')
if os.path.exists(caminho_resultados):
    print("📈 1. Evolução do Treinamento (Loss e mAP):")
    print("O que olhar para a banca: As linhas de 'loss' (erro) devem cair e as de 'mAP' (precisão) devem subir.")
    display(Image(filename=caminho_resultados, width=1200))
else:
    print("⚠️ Gráfico 'results.png' não encontrado. O treinamento já terminou?")

print("-" * 50)

# 2. Matriz de Confusão (Onde a IA mais erra)
caminho_matriz = os.path.join(pasta_resultados, 'confusion_matrix.png')
if os.path.exists(caminho_matriz):
    print("\n🧩 2. Matriz de Confusão:")
    print("O que olhar para a banca: Uma linha diagonal forte significa que a IA não está confundindo o 'B' com o '8', por exemplo.")
    display(Image(filename=caminho_matriz, width=1000))

print("-" * 50)

# 3. Predições de Validação (A prova visual)
caminho_predicao = os.path.join(pasta_resultados, 'val_batch0_pred.jpg')
if os.path.exists(caminho_predicao):
    print("\n👁️ 3. Amostra Visual (Como o YOLO enxerga as placas):")
    display(Image(filename=caminho_predicao, width=1200))

In [ ]:
import os
import shutil
from google.colab import files

# ====================================================
# 1. ESCOLHA DA PASTA DO TREINAMENTO
# ====================================================
# Olhando seu print, a "-3" é a do seu último treino.
# Se quiser baixar outra, é só mudar o número aqui.
NOME_DA_PASTA = 'ALPR_Vistoria_YOLOv8n-3'

caminho_pesos = f'/content/runs/detect/{NOME_DA_PASTA}/weights'
caminho_tfjs = f'{caminho_pesos}/best_web_model'
nome_zip = 'modelo_yolo_vistoria_tfjs'

print(f"📦 Preparando para empacotar o modelo da pasta: {NOME_DA_PASTA}...")

# ====================================================
# 2. COMPACTAÇÃO INTELIGENTE
# ====================================================
# O YOLO exporta o TFJS geralmente para uma pasta chamada 'best_web_model'
if os.path.exists(caminho_tfjs):
    shutil.make_archive(nome_zip, 'zip', caminho_tfjs)
    print("✅ Pasta do modelo Web (TFJS) zipada com sucesso!")
elif os.path.exists(caminho_pesos):
    # Se por acaso ele exportou solto na pasta weights, zipamos ela toda
    shutil.make_archive(nome_zip, 'zip', caminho_pesos)
    print("⚠️ Pasta web_model não achada. Zipando a pasta 'weights' inteira...")
else:
    print("❌ ERRO: Não achei a pasta. Verifique o NOME_DA_PASTA configurado.")

# ====================================================
# 3. DOWNLOAD AUTOMÁTICO
# ====================================================
if os.path.exists(f"{nome_zip}.zip"):
    print("⬇️ Solicitando download ao seu navegador... (Pode demorar alguns segundos, aguarde)")
    files.download(f"{nome_zip}.zip")